In [21]:
!pip install transformers -q

In [22]:
import logging

# Инициализация логгера
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

dh = logging.FileHandler('debug.log', encoding='utf-8')
dh.setLevel(logging.DEBUG)

eh = logging.FileHandler('error.log', encoding='utf-8')
eh.setLevel(logging.ERROR)

# Указываем время работы, название модуля, уровень логов и сообщение при отрабатывании
formatter = logging.Formatter('%(asctime)s-%(name)s-%(levelname)s-%(message)s')
dh.setFormatter(formatter)
eh.setFormatter(formatter)

logger.addHandler(eh)
logger.addHandler(dh)

In [23]:
import json
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import ast

with open('vacancies_details.json', 'r', encoding='utf-8') as f:
  vacancies = json.load(f)

#Удаляем дублирующиеся строки по id, которое содержится в ссылке, типа такого "https://hh.ru/vacancy/130841930?query" и вакансии без массива навыков
vac_df = pd.DataFrame(vacancies)
print(vac_df.shape)
vac_df["id"] = vac_df["link"].apply(lambda x:x.split('/')[-1].split('?')[0])
vac_df = vac_df.drop_duplicates(subset=["id"]).dropna(subset=["description"])
vac_df = vac_df[vac_df["skills"].apply(lambda x:len(x) > 0)]
print(vac_df.shape)

#Файл большой, поэтому может отваливаться загрузка
try:
  repos_df = pd.read_csv("repos_filtered.csv")
except:
  logger.error("Ошибка при загрузе")

(16195, 8)
(3364, 8)


In [24]:
from collections import Counter

def lang_bytes_dicts_to_list(langs):
  #Может быть nan в значениях
  if isinstance(langs, str):
    langs = dict(ast.literal_eval(langs))
    return [(x, langs[x]) for x in langs]

#Проходимся по всех языкам для пользователя и считаем суммарный байткод, после этого формируем список скилов
def lang_bytes_to_skills(langs):
    dct = {}
    result_skills = []
    sm_bytes = 0

    #Может быть nan в значениях
    for l in langs:
      if isinstance(l, tuple):
        if l[0] in dct:
          dct[l[0]] += l[1]
        else:
          dct[l[0]] = l[1]

        sm_bytes += l[1]

    #Взял только те языки, на которых больше 5% кода, чтобы убрать всякий мусор
    for key in dct.keys():
      dct[key] = round(100*dct[key]/sm_bytes, 2)
      if dct[key]>5:
        result_skills.append(key)

    if "Jupyter Notebook" in result_skills:
      result_skills.append("ML")

    return result_skills

repos_df["lang_list"] = repos_df["lang_bytes"].apply(lang_bytes_dicts_to_list)
exploded_repos = repos_df.explode("lang_list")

#Из словарей байткодов, превращаем языки в список навыков
users_skills_df = exploded_repos.groupby("owner_login").agg(lang_list=("lang_list", list))
users_skills_df["skills"] = users_skills_df["lang_list"].apply(lambda x:lang_bytes_to_skills(x))
users_skills_df = users_skills_df.reset_index()
users_skills_df = users_skills_df[users_skills_df['skills'].apply(lambda x:len(x)>0)]
users_skills_df

,owner_login,lang_list,skills
0,0niel,"[nan, (HTML, 2784), (Python, 30589), (Makefile...","[Dart, TypeScript, Java, C++, C]"
1,0xPh0enix,"[(C#, 89478), (C++, 6052), (C, 94), (C++, 1233...","[C#, C++]"
2,0xcds4r,"[nan, (C++, 2056311), (Makefile, 315506), (C, ...","[C++, C]"
3,1111alexandrr,"[nan, (Go, 12613941), (C, 673780), (JavaScript...","[Go, Rust]"
4,13xforever,"[(C#, 13258), (C#, 2490), (Rust, 1254), (C, 42...","[C#, C, C++, Python]"
...,...,...,...
978,zernovtechno,"[nan, (Python, 11412), (Python, 53112), (C++, ...","[C++, C#, C, HTML]"
979,zevlg,"[nan, (Rust, 520500), (Shell, 7198), (GLSL, 36...","[Roff, C, C++, Emacs Lisp]"
980,zhelnio,"[nan, (C, 166123), (Makefile, 100796), (C++, 1...",[C]
981,ziggi,"[(Pawn, 15036), (C, 2786), (PHP, 67844), (Java...","[Pawn, C, C++, TypeScript]"


In [25]:
model = SentenceTransformer('Nashhz/SBERT_KFOLD_Job_Descriptions_to_Skills')

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: Nashhz/SBERT_KFOLD_Job_Descriptions_to_Skills
DEBUG:httpcore.connection:close.started
DEBUG:httpcore.connection:close.complete
DEBUG:httpcore.connection:connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
DEBUG:httpcore.connection:connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x78c3f16e89e0>
DEBUG:httpcore.connection:start_tls.started ssl_context=<ssl.SSLContext object at 0x78c3ee136250> server_hostname='huggingface.co' timeout=10
DEBUG:httpcore.connection:start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x78c3f16e8b00>
DEBUG:httpcore.http11:send_request_headers.started request=<Request [b'HEAD']>
DEBUG:httpcore.http11:send_request_headers.complete
DEBUG:httpcore.http11:send_request_body.started request=<Request [b'HEA

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

DEBUG:httpcore.http11:send_request_headers.started request=<Request [b'HEAD']>
DEBUG:httpcore.http11:send_request_headers.complete
DEBUG:httpcore.http11:send_request_body.started request=<Request [b'HEAD']>
DEBUG:httpcore.http11:send_request_body.complete
DEBUG:httpcore.http11:receive_response_headers.started request=<Request [b'HEAD']>
DEBUG:httpcore.http11:receive_response_headers.complete return_value=(b'HTTP/1.1', 307, b'Temporary Redirect', [(b'Content-Type', b'text/plain; charset=utf-8'), (b'Content-Length', b'294'), (b'Connection', b'keep-alive'), (b'Date', b'Sat, 11 Apr 2026 00:13:29 GMT'), (b'Location', b'/api/resolve-cache/models/Nashhz/SBERT_KFOLD_Job_Descriptions_to_Skills/1f44fb6bb39d6124e3e4c689c079f55f9790ca3f/config.json?%2FNashhz%2FSBERT_KFOLD_Job_Descriptions_to_Skills%2Fresolve%2Fmain%2Fconfig.json=&etag=%22a8b60678a3b48d6be2f9e6de3855320831baa843%22'), (b'X-Powered-By', b'huggingface-moon'), (b'X-Request-Id', b'Root=1-69d99229-0f67764b459c6fe7554ef68c;732c9877-84a8-

In [ ]:
#Превращаем в вектора скиллы с гитхаба и вакансий, описания вакансий.
#Так как по документации модели, она не видела в процессе обучения меньше 7 токенов на описании и меньше 4 токенов на навыках, то
#Сделаем так чтобы при мэтчинге описания с навыками разработчика у нас длина 'предложения с навыками' была не меньше 4 токенов
#Длина навыков вакансий при мэтчинге с навыками c гитахаба будет не меньше 7 токенов
#https://huggingface.co/Nashhz/SBERT_KFOLD_Job_Descriptions_to_Skills
skills_embedings = []

for idx, (_, row) in enumerate(users_skills_df.iterrows()):
  skills_embedings.append({})
  txt = f"Навыки разработчика c Github: {", ".join(row["skills"])}"
  skills_embedings[idx]["skills_vector"] = model.encode(txt)
  skills_embedings[idx]["login"] = row["owner_login"]
  skills_embedings[idx]["skills"] = row["skills"]

logger.debug("Созданы вектора для навыков с гита")


vac_embedings = []

for idx, (_, row) in enumerate(vac_df.iterrows()):
  vac_embedings.append({})
  txt = f"Список ключевых навыков, которые содержатся в вакансии: {", ".join(row["skills"])}"
  vac_embedings[idx]["skills_vector"] = model.encode(txt)
  vac_embedings[idx]["desc_vector"] = model.encode(row["description"])
  vac_embedings[idx]["description"] = row["description"]
  vac_embedings[idx]["link"] = row["link"]
  vac_embedings[idx]["skills"] = row["skills"]

logger.debug("Созданы вектора для навыков с вакансией и описания")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
from sentence_transformers import util


def matching(vac_embedings, skills_embedings):
  matched_data=[]

  #Используем встроенный метод для косинусного сходства эмбедингов, util.cos_sim() возвращает значения в диапозоне [-1,1]
  #Далее сведем этот диапозон к [0,1], чтобы в итоге получился интерпретируемый результат
  for _, vac in enumerate(vac_embedings):
    for _, skills in enumerate(skills_embedings):
      skills_score = util.cos_sim(vac["skills_vector"], skills["skills_vector"]).item()
      desc_score = util.cos_sim(vac["desc_vector"], skills["skills_vector"]).item()
      score = 0.5*skills_score + 0.5*desc_score

      matched_data.append({
          "login":skills["login"],
          "link":vac["link"],
          "description":vac["description"],
          "score":score,
          "vacancy_skills":vac["skills"],
          "git_skills":skills["skills"]
      })

  logger.debug("Данные смэтчены")
  return matched_data

#Функция для приведения score к приемлимому виду в диапозоне [0,1]
#Сделаем min-max scaling и возведем в квадрат чтобы сделать больше разницу между значениями
#Так как диапозон значений примерно (0,1, 0,7), то нам нужно растянуть значения, для лучшей интерпритируемости
def scaling_scores(matched_data):
  mx, mn = float("-inf"), float("inf")

  for item in matched_data:
    mn = min(mn, item["score"])
    mx = max(mx, item["score"])

  for item in matched_data:
    item["score"] = ((item["score"]-mn)/(mx-mn))**2

  return matched_data

matched_data = matching(vac_embedings, skills_embedings)
matched_data = scaling_scores(matched_data)
matched_data[0]

In [ ]:
output_df = pd.DataFrame(matched_data)
np.round(output_df.describe(),2)

In [ ]:
import matplotlib.pyplot as plt

plt.hist(output_df[output_df['score']>0.7]["score"])
plt.show();

In [ ]:
# output_df.to_csv("matched_df.csv", index=False)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# !cp matched_df.csv /content/drive/MyDrive/

In [ ]:
#output_df.to_csv('matched_df.csv.gz', compression='gzip', index=False)
#Файл будет большой, 18 гигабайт не сжатый, поэтому открывать и скачивать только с гугл диска

In [ ]:
#!cp matched_df.csv.gz /content/drive/MyDrive/